# SageMaker 跟踪

>[Amazon SageMaker](https://aws.amazon.com/sagemaker/) 是一项完全托管的服务，用于快速轻松地构建、训练和部署机器学习 (ML) 模型。

>[Amazon SageMaker Experiments](https://docs.aws.amazon.com/sagemaker/latest/dg/experiments.html) 是 `Amazon SageMaker` 的一项功能，可让您组织、跟踪、比较和评估 ML 实验和模型版本。

本 Notebook 展示了如何使用 LangChain 回调将提示和其他 LLM 超参数记录和跟踪到 `SageMaker Experiments` 中。在此，我们使用不同的场景来展示此功能：

* **场景 1**: *单个 LLM* - 使用单个 LLM 模型根据给定提示生成输出的情况。
* **场景 2**: *顺序链* - 使用两个 LLM 模型的顺序链的情况。
* **场景 3**: *带工具的代理 (思维链)* - 除了 LLM 外，还使用多个工具（搜索和数学）的情况。

在本 Notebook 中，我们将创建一个实验来记录每个场景的提示。

## 安装与设置

In [ ]:
%pip install --upgrade --quiet  sagemaker
%pip install --upgrade --quiet  langchain-openai
%pip install --upgrade --quiet  google-search-results

首先，设置所需的 API 密钥

* OpenAI：https://platform.openai.com/account/api-keys (用于 OpenAI LLM 模型)
* Google SERP API：https://serpapi.com/manage-api-key (用于 Google 搜索工具)

In [ ]:
import os

## Add your API keys below
os.environ["OPENAI_API_KEY"] = "<ADD-KEY-HERE>"
os.environ["SERPAPI_API_KEY"] = "<ADD-KEY-HERE>"

In [ ]:
from langchain_community.callbacks.sagemaker_callback import SageMakerCallbackHandler

In [ ]:
from langchain.agents import initialize_agent, load_tools
from langchain.chains import LLMChain, SimpleSequentialChain
from langchain_core.prompts import PromptTemplate
from langchain_openai import OpenAI
from sagemaker.analytics import ExperimentAnalytics
from sagemaker.experiments.run import Run
from sagemaker.session import Session

## LLM 提示跟踪

This document outlines the process for tracking Large Language Model (LLM) prompts. This process is designed to ensure that all prompts sent to LLMs are logged, versioned, and easily retrievable for debugging, auditing, and performance analysis.

本文档概述了跟踪大型语言模型（LLM）提示的过程。此过程旨在确保发送给 LLM 的所有提示都经过记录、版本化，并且易于检索，以便进行调试、审计和性能分析。

### 1. Prompt Logging

All prompts sent to an LLM must be logged. The log entry should include the following information:

### 1. 提示日志记录

所有发送到 LLM 的提示都必须记录。日志条目应包含以下信息：

*   **Timestamp:** The date and time the prompt was sent.
    *   **时间戳：** 提示发送的日期和时间。
*   **User ID:** The identifier of the user who initiated the prompt (if applicable).
    *   **用户 ID：** 发起提示的用户标识符（如果适用）。
*   **Prompt Text:** The full text of the prompt.
    *   **提示文本：** 提示的完整文本。
*   **Model Used:** The specific LLM model that received the prompt (e.g., `gpt-4`, `claude-3-opus`).
    *   **使用的模型：** 接收提示的特定 LLM 模型（例如，`gpt-4`、`claude-3-opus`）。
*   **Request ID:** A unique identifier for the request.
    *   **请求 ID：** 请求的唯一标识符。
*   **Response Snippet:** A short snippet of the LLM's response for quick reference.
    *   **响应片段：** LLM 响应的简短片段，用于快速参考。
*   **Metadata:** Any relevant metadata associated with the prompt (e.g., session ID, conversation history ID).
    *   **元数据：** 与提示相关的任何元数据（例如，会话 ID、对话历史记录 ID）。

```json
{
  "timestamp": "2023-10-27T10:30:00Z",
  "userId": "user123",
  "promptText": "Translate the following sentence to French: 'Hello, how are you?'",
  "modelUsed": "gpt-3.5-turbo",
  "requestId": "req-abc-123",
  "responseSnippet": "Bonjour, comment ça va ?",
  "metadata": {
    "sessionId": "session-xyz",
    "conversationHistoryId": "conv-789"
  }
}
```

### 2. Prompt Versioning

Prompts should be versioned to track changes and ensure reproducibility.

### 2. 提示版本控制

应为提示进行版本控制，以跟踪更改并确保可复现性。

*   **Version Number:** A sequential number indicating the version of the prompt.
    *   **版本号：** 指示提示版本的顺序号。
*   **Change Log:** A description of the changes made in this version.
    *   **更改日志：** 对此版本所做更改的描述。
*   **Date Created:** The date the version was created.
    *   **创建日期：** 创建版本的日期。
*   **Author:** The person or system that created the version.
    *   **作者：** 创建该版本的个人或系统。

```json
{
  "promptId": "prompt-001",
  "versions": [
    {
      "versionNumber": 1,
      "promptText": "Translate the following sentence to French: 'Hello, how are you?'",
      "changeLog": "Initial version",
      "dateCreated": "2023-10-26T09:00:00Z",
      "author": "system"
    },
    {
      "versionNumber": 2,
      "promptText": "Translate the following sentence to French, using formal language: 'Hello, how are you?'",
      "changeLog": "Added requirement for formal language",
      "dateCreated": "2023-10-27T08:00:00Z",
      "author": "user123"
    }
  ]
}
```

### 3. Retrieval and Analysis

The logged and versioned prompts should be easily retrievable for analysis.

### 3. 检索与分析

应易于检索已记录和版本化的提示以进行分析。

*   **Search Functionality:** Ability to search prompts by keywords, user ID, model, or date range.
    *   **搜索功能：** 按关键字、用户 ID、模型或日期范围搜索提示的功能。
*   **Performance Metrics:** Track metrics such as response time, accuracy, and cost per prompt.
    *   **性能指标：** 跟踪响应时间、准确性和每个提示的成本等指标。
*   **Auditing:** Ability to audit prompt history for compliance and security purposes.
    *   **审计：** 能够审计提示历史记录以用于合规性和安全目的。

This tracking system will help ensure the responsible and efficient use of LLMs.

此跟踪系统将有助于确保 LLM 的负责任和高效使用。

In [ ]:
# LLM Hyperparameters
HPARAMS = {
    "temperature": 0.1,
    "model_name": "gpt-3.5-turbo-instruct",
}

# Bucket used to save prompt logs (Use `None` is used to save the default bucket or otherwise change it)
BUCKET_NAME = None

# Experiment name
EXPERIMENT_NAME = "langchain-sagemaker-tracker"

# Create SageMaker Session with the given bucket
session = Session(default_bucket=BUCKET_NAME)

### 场景 1 - LLM

In [ ]:
RUN_NAME = "run-scenario-1"
PROMPT_TEMPLATE = "tell me a joke about {topic}"
INPUT_VARIABLES = {"topic": "fish"}

In [ ]:
with Run(
    experiment_name=EXPERIMENT_NAME, run_name=RUN_NAME, sagemaker_session=session
) as run:
    # Create SageMaker Callback
    sagemaker_callback = SageMakerCallbackHandler(run)

    # Define LLM model with callback
    llm = OpenAI(callbacks=[sagemaker_callback], **HPARAMS)

    # Create prompt template
    prompt = PromptTemplate.from_template(template=PROMPT_TEMPLATE)

    # Create LLM Chain
    chain = LLMChain(llm=llm, prompt=prompt, callbacks=[sagemaker_callback])

    # Run chain
    chain.run(**INPUT_VARIABLES)

    # Reset the callback
    sagemaker_callback.flush_tracker()

### 场景 2 - 顺序链

In [ ]:
RUN_NAME = "run-scenario-2"

PROMPT_TEMPLATE_1 = """You are a playwright. Given the title of play, it is your job to write a synopsis for that title.
Title: {title}
Playwright: This is a synopsis for the above play:"""
PROMPT_TEMPLATE_2 = """You are a play critic from the New York Times. Given the synopsis of play, it is your job to write a review for that play.
Play Synopsis: {synopsis}
Review from a New York Times play critic of the above play:"""

INPUT_VARIABLES = {
    "input": "documentary about good video games that push the boundary of game design"
}

In [ ]:
with Run(
    experiment_name=EXPERIMENT_NAME, run_name=RUN_NAME, sagemaker_session=session
) as run:
    # Create SageMaker Callback
    sagemaker_callback = SageMakerCallbackHandler(run)

    # Create prompt templates for the chain
    prompt_template1 = PromptTemplate.from_template(template=PROMPT_TEMPLATE_1)
    prompt_template2 = PromptTemplate.from_template(template=PROMPT_TEMPLATE_2)

    # Define LLM model with callback
    llm = OpenAI(callbacks=[sagemaker_callback], **HPARAMS)

    # Create chain1
    chain1 = LLMChain(llm=llm, prompt=prompt_template1, callbacks=[sagemaker_callback])

    # Create chain2
    chain2 = LLMChain(llm=llm, prompt=prompt_template2, callbacks=[sagemaker_callback])

    # Create Sequential chain
    overall_chain = SimpleSequentialChain(
        chains=[chain1, chain2], callbacks=[sagemaker_callback]
    )

    # Run overall sequential chain
    overall_chain.run(**INPUT_VARIABLES)

    # Reset the callback
    sagemaker_callback.flush_tracker()

### 场景 3 - 代理与工具

In [ ]:
RUN_NAME = "run-scenario-3"
PROMPT_TEMPLATE = "Who is the oldest person alive? And what is their current age raised to the power of 1.51?"

In [ ]:
with Run(
    experiment_name=EXPERIMENT_NAME, run_name=RUN_NAME, sagemaker_session=session
) as run:
    # Create SageMaker Callback
    sagemaker_callback = SageMakerCallbackHandler(run)

    # Define LLM model with callback
    llm = OpenAI(callbacks=[sagemaker_callback], **HPARAMS)

    # Define tools
    tools = load_tools(["serpapi", "llm-math"], llm=llm, callbacks=[sagemaker_callback])

    # Initialize agent with all the tools
    agent = initialize_agent(
        tools, llm, agent="zero-shot-react-description", callbacks=[sagemaker_callback]
    )

    # Run agent
    agent.run(input=PROMPT_TEMPLATE)

    # Reset the callback
    sagemaker_callback.flush_tracker()

## 加载日志数据

一旦提示被记录下来，我们就可以轻松地加载它们并将其转换为 Pandas DataFrame，如下所示。

In [ ]:
# Load
logs = ExperimentAnalytics(experiment_name=EXPERIMENT_NAME)

# Convert as pandas dataframe
df = logs.dataframe(force_refresh=True)

print(df.shape)
df.head()

如上所示，实验中有三个运行（行）对应于每种场景。每次运行都会将提示和相关的 LLM 设置/超参数记录为 json，并保存在 s3 存储桶中。您可以随时加载和探索每个 json 路径中的日志数据。